# 🏛️ Model Kolektibilitas WP DSPC — 01 · Pre-processing (Level WP)

Notebook ini menyiapkan data untuk model **prediksi tingkat kolektibilitas** dengan unit analisis **per wajib pajak (NPWP16)**.

**Framing kasus (Skenario B — triase portofolio berjalan):**
> Dari WP dengan tunggakan *saat ini*, kolektibilitasnya Rendah/Sedang/Tinggi — supaya tim penagihan bisa memprioritaskan tindakan.

Konsekuensi desain:
- **Waktu prediksi = snapshot data (sekarang)** → semua kolom yang teramati sampai snapshot (`SETOR_*`, `NILAI_CAIR_HISTORIS`, `NILAI_SISA`, `JML_SURAT_*`, `FLAG_*`, riwayat tindakan) adalah **fitur sah** — mereka tersedia di momen model dipakai.
- Target `LABEL`: tingkat kolektibilitas 3 kelas — **0 = Rendah, 1 = Sedang, 2 = Tinggi** — diagregasi ke WP dengan **modus** antar ketetapannya (seri → label terburuk; dijelaskan di §4).
- Populasi: **seluruh hasil tarikan data** — populasi sudah dikontrol per tanggal tarikan oleh pengirim data, tidak ada filter tambahan.
- **Agregasi per WP**: nilai rupiah dijumlah, flag diambil maksimum, kronologi diambil titik ekstrem (utang tertua, daluwarsa terdekat, tindakan terbaru), ragam kasus dihitung count/nunique.
- Tanggal mentah (`TGL_*`) **tidak dipakai apa adanya**; dikonversi menjadi durasi/umur relatif snapshot.
- Split train–test **stratified biasa** — cukup aman karena kini 1 baris = 1 WP (tidak ada lagi baris kembar antar sisi seperti desain per ketetapan).

> ⚠️ **Kerahasiaan:** dataset RAHASIA, hanya diolah lokal (`dataset/` dikecualikan dari Git). Notebook berhenti di tahap pre-processing — pemodelan di notebook 02.

## ⚙️ Setup — Import & Konfigurasi

In [ ]:
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Reproducibility
RNG = 42
np.random.seed(RNG)
pd.set_option("display.max_columns", 60)

CONFIG = {
    # Sumber data
    "path_utama":    "../dataset/SAMPLE_DATA_ENRICH.csv",      # 1 baris = 1 ketetapan (STP/SKP)
    "path_metrics":  "../dataset/taxpayer_metrics_result.csv", # 1 baris = 1 WP (kepatuhan SPT)
    "path_customer": "../dataset/data_customer.json",          # faktur keluar (pelanggan)
    "path_supplier": "../dataset/data_supplier.json",          # faktur masuk (pemasok)
    "path_output":   "../dataset/processed/wp_features.csv",
    # Target
    "label_col": "LABEL",
    "label_names": {0: "Rendah", 1: "Sedang", 2: "Tinggi"},  # tingkat kolektibilitas
    # Parameter turunan fitur
    "ambang_daluwarsa_dekat_hari": 730,     # fitur: sisa daluwarsa <= 2 tahun
    "min_baris_kategori_ketetapan": 30,     # kode ketetapan < n baris -> "LAIN"
    # Split
    "test_size": 0.20,
}

print("Setup selesai ✅")

## 1. 📥 Memuat Data & Normalisasi Kunci NPWP

Empat sumber data dipertemukan lewat **NPWP 16 digit**.
Kunci dinormalkan menjadi string bersih 16 digit (`strip` + `zfill`) karena:
- banyak NPWP diawali nol (hilang bila dibaca sebagai angka),
- kolom `NPWP` di file metrik mengandung spasi kosong di belakang (fixed-width).

In [ ]:
# --- Dataset utama: satu baris = satu ketetapan (STP/SKP) + kolom LABEL ---
df = pd.read_csv(CONFIG["path_utama"], sep=";", dtype=str)
df.columns = df.columns.str.strip()

# --- Metrik kepatuhan SPT (satu baris = satu WP) ---
met = pd.read_csv(CONFIG["path_metrics"], dtype=str)

# --- Faktur pelanggan & pemasok (satu baris = satu WP) ---
cust = pd.DataFrame(json.load(open(CONFIG["path_customer"], encoding="utf-8")))
supp = pd.DataFrame(json.load(open(CONFIG["path_supplier"], encoding="utf-8")))


def norm_npwp(series: pd.Series) -> pd.Series:
    """Normalisasi NPWP -> string 16 digit tanpa spasi."""
    return series.astype(str).str.strip().str.zfill(16)


df["NPWP16"] = norm_npwp(df["NPWP16"])
met["NPWP"] = norm_npwp(met["NPWP"])
cust["NPWP"] = norm_npwp(cust["npwp"])
supp["NPWP"] = norm_npwp(supp["npwp"])

print(f"Ketetapan (baris) : {len(df):,}")
print(f"WP unik           : {df['NPWP16'].nunique():,}")
print(f"Metrik SPT        : {len(met):,} WP")
print(f"Faktur customer   : {len(cust):,} WP (tahun {cust['tahun'].unique().tolist()})")
print(f"Faktur supplier   : {len(supp):,} WP (tahun {supp['tahun'].unique().tolist()})")

In [ ]:
# Sanity check: coverage kunci antar sumber data
utama = set(df["NPWP16"])
print("Coverage kunci NPWP terhadap dataset utama:")
for nama, ref in [("metrik SPT", met), ("customer", cust), ("supplier", supp)]:
    s = set(ref["NPWP"])
    print(f"  {nama:12s}: match {len(utama & s):,} / {len(utama):,} WP")

## 2. 🧹 Pembersihan Level Baris (ketetapan)

Tindakan:
1. **Duplikat penuh** (baris identik 100%) dibuang.
2. Kolom teks di-`strip` (mis. `JENIS_KPP_BKM = 'P '`).
3. Kolom tanggal → `datetime` (`NaT` = tindakan belum pernah dilakukan).
4. Kolom angka (termasuk **`LABEL`**) → numerik.
5. Kategori dirapikan: kode KPP dipetakan ke nama, sektor ekonomi dari digit-1 kode KLU,
   dan **kode jenis ketetapan** diekstrak dari segmen ke-2 `NO_STPSKP` (mis. `106`/`107`).
6. Fitur baris `SELISIH_TAHUN_TERBIT` (tahun terbit ketetapan − tahun pajak) dihitung di sini
   agar bisa dirata-rata saat agregasi ke WP.

In [ ]:
n_awal = len(df)

# 1) Duplikat penuh dibuang
df = df.drop_duplicates()

# 2) Rapikan kolom teks
for c in df.select_dtypes(include=["object", "str"]).columns:
    df[c] = df[c].str.strip()

# 3) Tanggal -> datetime
DATE_COLS = [
    "TGL_PRODUK_HUKUM", "TGL_UTANG_DPT_DITAGIH", "TGL_INKRAH", "TGL_DALUWARSA",
    "TGL_TEGURAN", "TGL_PENYAMPAIAN_SP", "TGL_BAPS", "TGL_KMK_CEGAH", "TGL_SPRINDRA",
]
for c in DATE_COLS:
    df[c] = pd.to_datetime(df[c], errors="coerce")

# 4) Numerik (LABEL ikut dikonversi, harus 0/1/2 tanpa nilai aneh)
NUM_COLS = [
    "LABEL",
    "NILAI_STPSKP", "NILAI_SISA", "SETOR_SEBELUM_COLL_DATE", "SETOR_SEBELUM_TEGURAN",
    "SETOR_TEGURAN", "SETOR_PAKSA", "SETOR_SITA", "SETOR_CEGAH", "SETOR_SPRINDRA",
    "JML_SURAT_TEGURAN", "JML_SURAT_PAKSA", "NILAI_CAIR_HISTORIS",
    "FG_INKRAH_CTX", "FLAG_PERNAH_DISITA", "FLAG_PERNAH_BLOKIR", "FLAG_RESPON_PENAGIHAN",
    "TH_PJK", "THN_DALUARSA",
]
for c in NUM_COLS:
    df[c] = pd.to_numeric(df[c], errors="coerce")
assert df["LABEL"].isin([0, 1, 2]).all(), "LABEL berisi nilai di luar {0,1,2}!"

# 5) Kategori
df["JENIS_KPP_BKM"] = df["JENIS_KPP_BKM"].replace(
    {"P": "PRATAMA", "M": "MADYA", "B": "BESAR", "K": "KHUSUS"}
)
df["SEKTOR_KLU"] = df["KD_KLU"].str.zfill(5).str[0].map({
    "0": "PERTANIAN", "1": "PERTAMBANGAN", "2": "INDUSTRI", "3": "ENERGI/AIR",
    "4": "KONSTRUKSI", "5": "PERDAGANGAN", "6": "TRANSPORTASI", "7": "INFORMASI",
    "8": "JASA", "9": "JASA_LAIN",
}).fillna("LAINNYA")

# Kode jenis ketetapan dari segmen ke-2 NO_STPSKP ("00025/106/17/..." -> "106")
df["KODE_JENIS_KETETAPAN"] = df["NO_STPSKP"].str.split("/").str[1]
jarang = df["KODE_JENIS_KETETAPAN"].value_counts()
df["KODE_JENIS_KETETAPAN"] = df["KODE_JENIS_KETETAPAN"].where(
    df["KODE_JENIS_KETETAPAN"].isin(jarang[jarang >= CONFIG["min_baris_kategori_ketetapan"]].index),
    "LAIN",
)

# 6) Fitur baris (dihitung sebelum agregasi)
df["SELISIH_TAHUN_TERBIT"] = df["TGL_PRODUK_HUKUM"].dt.year - df["TH_PJK"]

print(f"Baris: {n_awal:,} -> {len(df):,} (duplikat penuh dibuang: {n_awal - len(df)})")
print("\nLABEL              :", df["LABEL"].value_counts().sort_index().to_dict())
print("STS_WP             :", df["STS_WP"].value_counts().to_dict())
print("JENIS_KPP_BKM      :", df["JENIS_KPP_BKM"].value_counts().to_dict())
print("JENIS_WP           :", df["JENIS_WP"].value_counts().to_dict())
print("KODE_JENIS_KETETAPAN :", df["KODE_JENIS_KETETAPAN"].value_counts().to_dict())
print("SEKTOR_KLU         :", df["SEKTOR_KLU"].value_counts().to_dict())

In [ ]:
# Kualitas data: kolom bermasalah (banyak missing / konstan)
info = pd.DataFrame({
    "dtype": df.dtypes.astype(str),
    "missing": df.isna().sum(),
    "n_unique": df.nunique(dropna=True),
})
info["pct_missing"] = (info["missing"] / len(df) * 100).round(1)
KOLOM_LEMAH = info[(info["pct_missing"] == 100) | (info["n_unique"] <= 1)].index.tolist()
display(info.loc[KOLOM_LEMAH])
print("-> Kolom kosong/konstan (tidak informatif):", KOLOM_LEMAH)

## 3. 📸 Snapshot & Profil Populasi

Populasi **tidak difilter apa pun** — populasi sudah ditentukan oleh kriteria tarikan data
(per tanggal tarikan) di sisi sumber. Sel ini hanya menetapkan **snapshot observasi**
(titik "sekarang" untuk semua fitur durasi) dan menampilkan profil populasi.

In [ ]:
SNAPSHOT = df["TGL_UTANG_DPT_DITAGIH"].max()
print(f"Snapshot observasi : {SNAPSHOT.date()}")

n_wp = df["NPWP16"].nunique()
n_multi = (df.groupby("NPWP16").size() > 1).sum()
n_outstanding = (df["NILAI_SISA"] > 0).sum()

print(f"\nPopulasi      : {len(df):,} ketetapan | {n_wp:,} WP ({n_multi} WP memiliki >1 ketetapan)")
print("Status WP     :", df.drop_duplicates('NPWP16')["STS_WP"].value_counts().to_dict())
print(f"Ketetapan outstanding (NILAI_SISA > 0): {n_outstanding:,} | lunas: {len(df)-n_outstanding:,}")
print(f"\nProfil (informatif):")
print(f"  Umur inkrah (hari) : median {(SNAPSHOT - df['TGL_INKRAH']).dt.days.median():.0f} "
      f"| maks {(SNAPSHOT - df['TGL_INKRAH']).dt.days.max():.0f}")
print(f"  Sisa daluwarsa <= 2 thn: {((df['TGL_DALUWARSA'] - SNAPSHOT).dt.days <= 730).sum()} ketetapan")

## 4. 📦 Agregasi per WP (NPWP16)

Satu WP bisa memiliki beberapa ketetapan → semua kolom diagregasi ke **level WP**:

| Kelompok | Aturan | Kolom |
|---|---|---|
| Identitas | `first` | `NAMA_WP`, `STS_WP`, `JENIS_WP`, `JENIS_KPP_BKM`, `KD_KANWIL`, `KD_KLU`, `SEKTOR_KLU` |
| Nilai rupiah | `sum` | `NILAI_STPSKP` → `TOTAL_TUNGGAKAN_POKOK`, `NILAI_SISA`, `NILAI_CAIR_HISTORIS`, `SETOR_*` |
| Flag | `max` | `FLAG_PERNAH_DISITA`, `FLAG_PERNAH_BLOKIR`, `FLAG_RESPON_PENAGIHAN` |
| Kronologi awal | `min` | `TGL_INKRAH`, `TGL_UTANG_DPT_DITAGIH` (tertua), `TGL_DALUWARSA` (terdekat) |
| Kronologi tindakan | `max` | `TGL_TEGURAN`, `TGL_PENYAMPAIAN_SP`, `TGL_BAPS` (terbaru) |
| Ragam kasus | `count` / `nunique` / `mean` | `JML_KETETAPAN`, `JML_JENIS_PAJAK`, `JML_JENIS_KETETAPAN`, `RATA_SELISIH_TAHUN_TERBIT`, `JML_SURAT_PAKSA` (sum) |
| **Target** | **modus** | `LABEL` — label mayoritas antar ketetapan; **seri → label terburuk (terendah)** |

**Kenapa modus untuk LABEL?** Kolektibilitas WP mencerminkan keseluruhan portofolio tunggakannya;
label mayoritas adalah representasi paling wajar. Bila dua label berimbang, diambil yang
terburuk (konservatif untuk skenario triase). Jumlah WP ber-label campuran dilaporkan di bawah.

In [ ]:
AGG = {
    # Target: modus antar ketetapan; seri -> terburuk (modus pandas mengembalikan nilai terkecil lebih dulu)
    "LABEL": ("LABEL", lambda s: s.mode().iloc[0]),
    # Identitas
    "NAMA_WP": ("NAMA_WP", "first"),
    "STS_WP": ("STS_WP", "first"),
    "JENIS_WP": ("JENIS_WP", "first"),
    "JENIS_KPP_BKM": ("JENIS_KPP_BKM", "first"),
    "KD_KANWIL": ("KD_KANWIL", "first"),
    "KD_KLU": ("KD_KLU", "first"),
    "SEKTOR_KLU": ("SEKTOR_KLU", "first"),
    # Nilai rupiah: dijumlah
    "TOTAL_TUNGGAKAN_POKOK": ("NILAI_STPSKP", "sum"),
    "NILAI_TUNGGAKAN_SISA": ("NILAI_SISA", "sum"),
    "TOTAL_NILAI_CAIR": ("NILAI_CAIR_HISTORIS", "sum"),
    "SETOR_SEBELUM_COLL_DATE": ("SETOR_SEBELUM_COLL_DATE", "sum"),
    "SETOR_SEBELUM_TEGURAN": ("SETOR_SEBELUM_TEGURAN", "sum"),
    "SETOR_TEGURAN": ("SETOR_TEGURAN", "sum"),
    "SETOR_PAKSA": ("SETOR_PAKSA", "sum"),
    "SETOR_SITA": ("SETOR_SITA", "sum"),
    "SETOR_CEGAH": ("SETOR_CEGAH", "sum"),
    "SETOR_SPRINDRA": ("SETOR_SPRINDRA", "sum"),
    # Flag: pernah = maksimum
    "FLAG_PERNAH_DISITA": ("FLAG_PERNAH_DISITA", "max"),
    "FLAG_PERNAH_BLOKIR": ("FLAG_PERNAH_BLOKIR", "max"),
    "FLAG_RESPON_PENAGIHAN": ("FLAG_RESPON_PENAGIHAN", "max"),
    # Kronologi: utang tertua & tenggat terdekat
    "TGL_INKRAH_PERTAMA": ("TGL_INKRAH", "min"),
    "TGL_UTANG_PERTAMA": ("TGL_UTANG_DPT_DITAGIH", "min"),
    "TGL_DALUWARSA_TERDEKAT": ("TGL_DALUWARSA", "min"),
    # Kronologi tindakan: terbaru
    "TGL_TEGURAN_TERBARU": ("TGL_TEGURAN", "max"),
    "TGL_PENYAMPAIAN_TERBARU": ("TGL_PENYAMPAIAN_SP", "max"),
    "TGL_BAPS_TERBARU": ("TGL_BAPS", "max"),
    # Ragam kasus
    "JML_KETETAPAN": ("NO_STPSKP", "count"),
    "JML_JENIS_PAJAK": ("JENIS_PAJAK", "nunique"),
    "JML_JENIS_KETETAPAN": ("KODE_JENIS_KETETAPAN", "nunique"),
    "JML_SURAT_PAKSA": ("JML_SURAT_PAKSA", "sum"),
    "RATA_SELISIH_TAHUN_TERBIT": ("SELISIH_TAHUN_TERBIT", "mean"),
}

wp = df.groupby("NPWP16", as_index=False).agg(**AGG)

# Laporan agregasi label
n_label_campur = (df.groupby("NPWP16")["LABEL"].nunique() > 1).sum()
print(f"WP teragregasi      : {len(wp):,} baris × {wp.shape[1]} kolom")
print(f"WP ber-label campur : {n_label_campur} WP (label WP = modus; seri -> terburuk)")
print(f"Distribusi LABEL WP :", wp["LABEL"].value_counts().sort_index().to_dict())
wp[["NPWP16", "NAMA_WP", "STS_WP", "JML_KETETAPAN", "TOTAL_TUNGGAKAN_POKOK",
    "NILAI_TUNGGAKAN_SISA", "TOTAL_NILAI_CAIR", "LABEL"]].head()

## 5. 🔗 Penggabungan Data Eksternal

Metrik SPT & faktur customer/supplier memang level WP → join **1-ke-1** bersih (tanpa broadcast).

In [ ]:
met_f = met.rename(columns={
    "NPWP": "NPWP16",
    "rasio_lapor_spt_3thn": "RASIO_LAPOR_SPT_3THN",
    "flag_lapor_spt_terakhir": "FLAG_LAPOR_SPT_TERAKHIR",
    "peredaran_bruto": "PEREDARAN_BRUTO",
})
for c in ["RASIO_LAPOR_SPT_3THN", "FLAG_LAPOR_SPT_TERAKHIR", "PEREDARAN_BRUTO"]:
    met_f[c] = pd.to_numeric(met_f[c], errors="coerce")

cust_f = cust[["NPWP", "jml_customer", "total_jml_dpp", "total_jml_faktur"]].rename(columns={
    "NPWP": "NPWP16",
    "jml_customer": "JML_CUSTOMER", "total_jml_dpp": "DPP_CUSTOMER", "total_jml_faktur": "FAKTUR_CUSTOMER",
})
supp_f = supp[["NPWP", "jml_supplier", "total_jml_dpp", "total_jml_faktur"]].rename(columns={
    "NPWP": "NPWP16",
    "jml_supplier": "JML_SUPPLIER", "total_jml_dpp": "DPP_SUPPLIER", "total_jml_faktur": "FAKTUR_SUPPLIER",
})
for f in (cust_f, supp_f):
    for c in f.columns[1:]:
        f[c] = pd.to_numeric(f[c], errors="coerce")

wp = (
    wp.merge(met_f, on="NPWP16", how="left", validate="one_to_one")
      .merge(cust_f, on="NPWP16", how="left", validate="one_to_one")
      .merge(supp_f, on="NPWP16", how="left", validate="one_to_one")
)

cek_na = wp[["RASIO_LAPOR_SPT_3THN", "JML_CUSTOMER", "JML_SUPPLIER"]].isna().sum()
print("Nilai hilang setelah penggabungan:", cek_na.to_dict())
print(f"Dimensi WP: {wp.shape}")
wp[["NPWP16", "JML_KETETAPAN", "TOTAL_TUNGGAKAN_POKOK", "RASIO_LAPOR_SPT_3THN",
    "JML_CUSTOMER", "JML_SUPPLIER"]].head()

## 6. 🛠️ Feature Engineering (as-of-snapshot, level WP)

Semua fitur dihitung relatif terhadap **snapshot** (waktu prediksi triase):

**a. Durasi** — tanggal mentah dikonversi ke umur (hari); tindakan yang belum pernah terjadi → `NaN` + flag keberadaan:
- `UMUR_TUNGGAKAN_HARI` / `UMUR_INKRAH_HARI` (sejak utang/inkrah tertua), `SISA_DALUWARSA_HARI` (tenggat terdekat) + flag ≤ 2 tahun,
- `HARI_SEJAK_TEGURAN/SP/BAPS` (sejak tindakan terbaru) + flag ada/tidak.

**b. Rasio keuangan WP** — sinyal triase paling langsung, dihitung dari total WP:
- `RASIO_CAIR` = total pencairan / total pokok, `RASIO_SISA` = total sisa / total pokok (0 = lunas, 1 = belum tersentuh),
- `RASIO_TUNGGAKAN_PEREDARAN` vs peredaran usaha (peredaran 0 → `NaN` + flag).

**c. Transformasi `log1p`** untuk nilai rupiah yang sangat miring + **total mitra** (customer + supplier).

In [ ]:
# ---------- a. Durasi relatif snapshot ----------
wp["UMUR_TUNGGAKAN_HARI"] = (SNAPSHOT - wp["TGL_UTANG_PERTAMA"]).dt.days
wp["UMUR_INKRAH_HARI"] = (SNAPSHOT - wp["TGL_INKRAH_PERTAMA"]).dt.days
wp["SISA_DALUWARSA_HARI"] = (wp["TGL_DALUWARSA_TERDEKAT"] - SNAPSHOT).dt.days
wp["FLAG_DALUWARSA_DEKAT"] = (wp["SISA_DALUWARSA_HARI"] <= CONFIG["ambang_daluwarsa_dekat_hari"]).astype(int)

for kol, baru in [("TGL_TEGURAN_TERBARU", "TEGURAN"), ("TGL_PENYAMPAIAN_TERBARU", "SP"), ("TGL_BAPS_TERBARU", "BAPS")]:
    wp[f"HARI_SEJAK_{baru}"] = (SNAPSHOT - wp[kol]).dt.days          # NaN jika belum pernah
    wp[f"FLAG_{baru}_ADA"] = wp[kol].notna().astype(int)

# ---------- b. Rasio keuangan WP ----------
wp["RASIO_CAIR"] = (wp["TOTAL_NILAI_CAIR"] / wp["TOTAL_TUNGGAKAN_POKOK"].replace(0, np.nan)).clip(0, 2)
wp["RASIO_SISA"] = (wp["NILAI_TUNGGAKAN_SISA"] / wp["TOTAL_TUNGGAKAN_POKOK"].replace(0, np.nan)).clip(0, 1)

wp["FLAG_PEREDARAN_NOL"] = (wp["PEREDARAN_BRUTO"] <= 0).astype(int)
wp["RASIO_TUNGGAKAN_PEREDARAN"] = np.where(
    wp["PEREDARAN_BRUTO"] > 0,
    wp["TOTAL_TUNGGAKAN_POKOK"] / wp["PEREDARAN_BRUTO"],
    np.nan,
)

# ---------- c. log1p & jejaring ----------
for src, dst in [
    ("TOTAL_TUNGGAKAN_POKOK", "LOG_TUNGGAKAN_POKOK"),
    ("TOTAL_NILAI_CAIR", "LOG_NILAI_CAIR"),
    ("PEREDARAN_BRUTO", "LOG_PEREDARAN_BRUTO"),
    ("DPP_CUSTOMER", "LOG_DPP_CUSTOMER"),
    ("DPP_SUPPLIER", "LOG_DPP_SUPPLIER"),
]:
    wp[dst] = np.log1p(wp[src].clip(lower=0))
wp["TOTAL_MITRA"] = wp["JML_CUSTOMER"] + wp["JML_SUPPLIER"]

wp[["UMUR_TUNGGAKAN_HARI", "SISA_DALUWARSA_HARI", "HARI_SEJAK_TEGURAN", "HARI_SEJAK_SP",
    "RASIO_CAIR", "RASIO_SISA", "RASIO_TUNGGAKAN_PEREDARAN", "TOTAL_MITRA"]].describe().T.round(2)

## 7. 🏷️ Target — `LABEL` (Tingkat Kolektibilitas, level WP)

`LABEL` WP = modus label ketetapannya ∈ {0 = Rendah, 1 = Sedang, 2 = Tinggi}.

Tabel verifikasi memperlihatkan penilaian kolektibilitas selaras dengan perilaku pembayaran
yang teramati (rasio pencairan & respon meningkat dari Rendah → Tinggi) — masuk akal secara
domain.

In [ ]:
LAB = CONFIG["label_col"]

# Distribusi kelas
dist = wp[LAB].value_counts().sort_index()
pct = (dist / len(wp) * 100).round(1)
tabel_label = pd.DataFrame(
    {"kolektibilitas": [CONFIG["label_names"][k] for k in dist.index],
     "jumlah": dist, "pct": pct}
).rename_axis(LAB)
display(tabel_label)

# Keterkaitan LABEL dengan perilaku teramati
eda = wp[[LAB]].assign(
    RASIO_CAIR=wp["RASIO_CAIR"], RESPON=wp["FLAG_RESPON_PENAGIHAN"],
    PAKSA=wp["JML_SURAT_PAKSA"], RASIO_SISA=wp["RASIO_SISA"],
)
display(eda.groupby(LAB).agg(
    rasio_cair_mean=("RASIO_CAIR", "mean"),
    rasio_sisa_mean=("RASIO_SISA", "mean"),
    respon_rate=("RESPON", "mean"),
    rata_surat_paksa=("PAKSA", "mean"),
).round(3).rename(index=CONFIG["label_names"]))

fig, ax = plt.subplots(figsize=(5, 3))
dist.plot(kind="bar", ax=ax, color=["#c44e52", "#dd8452", "#55a868"])
ax.set_xticklabels([f"{k} = {CONFIG['label_names'][k]}" for k in dist.index], rotation=0)
ax.set_ylabel("Jumlah WP")
ax.set_title("Distribusi target LABEL (level WP)")
for i, v in enumerate(dist):
    ax.text(i, v, f"{v:,}", ha="center", va="bottom", fontsize=9)
plt.tight_layout()
plt.show()

## 8. 🧮 Pre-processing untuk Machine Learning

Kolom dikelompokkan berdasarkan perannya:

| Peran | Kolom | Nasib |
|---|---|---|
| `ID_COLS` | `NPWP16`, `NAMA_WP`, `KD_KLU` | disimpan untuk pelacakan, **bukan** fitur |
| Target | `LABEL` | dipisah sebagai `y` |
| `TGL_*` mentah | semua kolom tanggal | **tidak dipakai langsung** — sudah dikonversi ke durasi/flag di §6 |
| Konstan/kosong | `FG_INKRAH_CTX`, `JML_SURAT_TEGURAN`, `NO/TGL_KMK_CEGAH`, `NO/TGL_SPRINDRA` | dibuang (nol informasi) |
| Kategori | `STS_WP`, `JENIS_WP`, `JENIS_KPP_BKM`, `KD_KANWIL`, `SEKTOR_KLU` | one-hot encoding |
| Numerik | sisanya (termasuk `SETOR_*`, total pencairan/sisa, `FLAG_*`, durasi) | imputasi median + standardisasi |

Catatan split: karena kini **1 baris = 1 WP**, split stratified biasa sudah bebas dari
korelasi antar-baris (group split tidak lagi diperlukan). Imputasi & encoding tetap dibungkus
`ColumnTransformer` yang **fit hanya pada train**.

In [ ]:
ID_COLS = ["NPWP16", "NAMA_WP", "KD_KLU"]
RAW_DATE_COLS = ["TGL_INKRAH_PERTAMA", "TGL_UTANG_PERTAMA", "TGL_DALUWARSA_TERDEKAT",
                 "TGL_TEGURAN_TERBARU", "TGL_PENYAMPAIAN_TERBARU", "TGL_BAPS_TERBARU"]

CAT_FEATURES = ["STS_WP", "JENIS_WP", "JENIS_KPP_BKM", "KD_KANWIL", "SEKTOR_KLU"]
NUM_FEATURES = [
    # Portofolio tunggakan WP
    "TOTAL_TUNGGAKAN_POKOK", "LOG_TUNGGAKAN_POKOK", "NILAI_TUNGGAKAN_SISA",
    "JML_KETETAPAN", "JML_JENIS_PAJAK", "JML_JENIS_KETETAPAN", "RATA_SELISIH_TAHUN_TERBIT",
    # Kondisi pembayaran & penagihan (as-of-snapshot)
    "TOTAL_NILAI_CAIR", "LOG_NILAI_CAIR", "RASIO_CAIR", "RASIO_SISA",
    "SETOR_SEBELUM_COLL_DATE", "SETOR_SEBELUM_TEGURAN", "SETOR_TEGURAN",
    "SETOR_PAKSA", "SETOR_SITA", "SETOR_CEGAH", "SETOR_SPRINDRA",
    "JML_SURAT_PAKSA",
    "FLAG_PERNAH_DISITA", "FLAG_PERNAH_BLOKIR", "FLAG_RESPON_PENAGIHAN",
    # Durasi & keberadaan tindakan
    "UMUR_TUNGGAKAN_HARI", "UMUR_INKRAH_HARI", "SISA_DALUWARSA_HARI", "FLAG_DALUWARSA_DEKAT",
    "HARI_SEJAK_TEGURAN", "HARI_SEJAK_SP", "HARI_SEJAK_BAPS",
    "FLAG_TEGURAN_ADA", "FLAG_SP_ADA", "FLAG_BAPS_ADA",
    # Kepatuhan & aktivitas ekonomi
    "RASIO_LAPOR_SPT_3THN", "FLAG_LAPOR_SPT_TERAKHIR",
    "PEREDARAN_BRUTO", "LOG_PEREDARAN_BRUTO", "FLAG_PEREDARAN_NOL",
    "RASIO_TUNGGAKAN_PEREDARAN",
    "JML_CUSTOMER", "DPP_CUSTOMER", "LOG_DPP_CUSTOMER", "FAKTUR_CUSTOMER",
    "JML_SUPPLIER", "DPP_SUPPLIER", "LOG_DPP_SUPPLIER", "FAKTUR_SUPPLIER",
    "TOTAL_MITRA",
]
FEATURES = NUM_FEATURES + CAT_FEATURES

# Verifikasi: tidak ada kolom tanggal mentah / ID / target yang lolos menjadi fitur
assert LAB not in FEATURES
assert not any(c in FEATURES for c in ID_COLS + RAW_DATE_COLS)
print(f"Fitur numerik    ({len(NUM_FEATURES)}): {NUM_FEATURES}")
print(f"Fitur kategorikal ({len(CAT_FEATURES)}): {CAT_FEATURES}")

X = wp[FEATURES].copy()
y = wp[LAB].copy()
print(f"\nX: {X.shape} | y: {y.shape}")

In [ ]:
from sklearn.model_selection import train_test_split
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder, StandardScaler

# 1 baris = 1 WP -> split stratified biasa (proporsi 3 kelas terjaga, tanpa risiko baris kembar)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=CONFIG["test_size"], stratify=y, random_state=RNG,
)

dist_train = y_train.value_counts(normalize=True).sort_index() * 100
dist_test = y_test.value_counts(normalize=True).sort_index() * 100
print(f"Train: {X_train.shape}  |  Test: {X_test.shape}")
print("\nProporsi kelas (train vs test, %):")
display(pd.DataFrame({"train": dist_train.round(1), "test": dist_test.round(1)}).rename(index=CONFIG["label_names"]))

preprocessor = ColumnTransformer(
    transformers=[
        ("num", Pipeline([
            ("imputer", SimpleImputer(strategy="median")),
            ("scaler", StandardScaler()),
        ]), NUM_FEATURES),
        ("cat", Pipeline([
            ("imputer", SimpleImputer(strategy="most_frequent")),
            ("onehot", OneHotEncoder(handle_unknown="ignore", sparse_output=False)),
        ]), CAT_FEATURES),
    ],
    remainder="drop",
)

X_train_prep = preprocessor.fit_transform(X_train)  # fit HANYA pada train
X_test_prep = preprocessor.transform(X_test)

nama_fitur = preprocessor.get_feature_names_out()
print(f"\nSetelah pre-processing: train {X_train_prep.shape}, test {X_test_prep.shape}")
print(f"Nilai hilang di matriks hasil: train={np.isnan(X_train_prep).sum()}, test={np.isnan(X_test_prep).sum()}")
print("Contoh nama fitur:", list(nama_fitur[:5]), "...")

pd.DataFrame(X_train_prep, columns=nama_fitur).iloc[:5, :8]

## 9. 💾 Simpan Hasil

In [ ]:
import os

os.makedirs(os.path.dirname(CONFIG["path_output"]), exist_ok=True)
wp.to_csv(CONFIG["path_output"], index=False)

dist = wp[LAB].value_counts().sort_index()
print(f"Dataset level WP tersimpan : {CONFIG['path_output']}")
print(f"Dimensi                   : {wp.shape[0]:,} WP × {wp.shape[1]} kolom")
print("Distribusi LABEL          :",
      " | ".join(f"{CONFIG['label_names'][k]}={v:,}" for k, v in dist.items()))

## 📌 Ringkasan & Langkah Berikutnya

**Hasil notebook ini (Skenario B — triase portofolio, level WP):**
- Unit analisis: **1 baris = 1 WP** (2.873 WP hasil agregasi 3.000 ketetapan) — tanpa filter populasi.
- Agregasi: rupiah `sum`, flag `max`, kronologi `min`/`max`, ragam `count`/`nunique`; `LABEL` = **modus** antar ketetapan (seri → terburuk).
- Waktu prediksi = **snapshot** → fitur as-of-snapshot sah (`SETOR_*`, total pencairan/sisa, `FLAG_*`, durasi tindakan).
- Split stratified biasa (1 baris = 1 WP) + preprocessor siap pakai (median-impute + scaling, one-hot) — fit hanya di train.

**Notebook 02 — modelling:**
- [ ] Baseline (Logistic Regression / Random Forest / LightGBM) — metrik **macro-F1** + recall per kelas
- [ ] Cross-validation stratified biasa (`StratifiedKFold`)
- [ ] Analisis performa per segmen `STS_WP` (AKTIF vs NE)
- [ ] Interpretasi (feature importance / SHAP) — verifikasi sinyal triase masuk akal secara domain
- [ ] Skor langsung = prioritas WP untuk tim penagihan (tidak perlu agregasi tambahan)
- [ ] Pertimbangan deploy: model di-train pada mix WP lunas+outstanding; saat triase skor diterapkan pada WP outstanding — pantau drift